# Decision dump run
Runs the standard 6 queries on every photo and saves **every candidate of every query** (place, coarse z, inliers, shift, heading, scale) plus the gate row. The decision can then be re-made offline in other ways (voting across queries, agreement radius, averaged position) without running the GPU again.

Settings: env `GATE_DATA` (official / nudt), `NUDT_SEED`, `N_NUDT`, `EXCLUDE_SEED` (skip another seed's photos, for a fresh sample).

In [1]:
import os, sys, math, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
DATA = os.environ.get("GATE_DATA", "official")           # "official" (d1 photos; the map comes from src/header/map_source.py) or "nudt" (Changsha test photos)
MAP = "new"                 # base map of the official photos: "new" (both scenes), "new_931", "new_921" or "old"
MAP_MARGIN_M = None         # metres of map kept around the photos (None = 6000 from map_source.py; the search disc is 5500 m)
POOL_PHOTOS = 500           # photos loaded once; the seeds pick from them
SEEDS = [11]        # three random seeds, side by side: each picks its own photos
N_PER_SEED = 150            # photos per seed (a multiple of 3: the same number from each height group). 150 x 3 seeds covers about 330 of the ~500 photos, so the seeds overlap and the 'all' column is the one to trust for the size of the gain
FOUND_SIGMA = 4.0           # a place counts as found when it is within this many sigma of the truth

# the queries: every photo is drawn once per (heading, scale factor)
HEADINGS = [-5.0, 0.0, 5.0]   # degrees added to the yaw (Patch v0.5.6: the needed corrections are within +-10)
SCALE_FACTORS = [1.0, 1.05]       # the projection scale is multiplied by each of these; [1.0, 1.05] doubles the queries (level 7 photos preferred 1.05)

# per query: the same pipeline as Single_Image_Run
COARSE = "edge_zncc"        # "edge_zncc" (one map, cheap: the patch recommends it for the hypotheses) or "fused" (six maps, about 9 times that cost)
USE_SEMANTIC_LAYER = True   # DINO re-orders the K coarse places of each query
K_PER_QUERY = 50            # K: places the coarse search returns for each query
M_PER_QUERY = 10            # M: places of each query that go to LoFTR (5 gave the same score as 25 on the development photos)
DINO_WEIGHT = None          # None = 100 (DINO decides the order inside the K)
GAMMA = 1.0                 # the answer is the candidate with the most  inliers + GAMMA * (DINO z-score) among ALL queries
MATCHER = "loftr_minima"    # "lightglue", "loftr_minima" or "roma_minima"
MATCHER_CONFIDENCE = 0.22   # LoFTR only: matches below this confidence are dropped (0.2 = kornia's default; lower lets weak, possibly false matches through)
LOFTR_COARSE_THRESHOLD = 0.21  # LoFTR only: its own cut on the coarse match probability
RANSAC_PIXELS = None        # None = 3.0
MIN_INLIERS = 8             # answer bar

# the variants that are scored from the same run: name -> (headings, scale factors); a variant needs all its queries to exist
VARIANTS = {
    "plain (heading 0)": ([0.0], [1.0]),
    "3 headings (-5, 0, +5)": ([-5.0, 0.0, 5.0], [1.0]),
    "5 headings (step 5)": ([-10.0, -5.0, 0.0, 5.0, 10.0], [1.0]),
    "all headings": (HEADINGS, [1.0]),
    "all headings x all scales": (HEADINGS, SCALE_FACTORS),
}
# the same candidates are also scored with a cost for the heading: each candidate pays  penalty * |heading|  inliers (inliers per degree) when the best one is chosen.
# 0 = the best candidate wins whatever its heading (the variants above). These are scored on the "all headings" queries from the same run, nothing is run twice.
HEADING_PENALTIES = []     # 0.5 means a candidate 10 degrees off needs 5 more inliers than one at 0 degrees
plt.rcParams["figure.dpi"] = 90

assert len(SEEDS) >= 1 and N_PER_SEED >= 3, "need at least one seed and 3 photos per seed"
assert 0.0 in HEADINGS and 1.0 in SCALE_FACTORS, "the plain query (heading 0, scale factor 1) is the baseline and must be in the lists"

In [2]:
from heading_query_pipeline import HeadingQueryPipeline        # first import: it saves the development-set settings before NUDT changes them
from header import settings
from header.sampling import pick_sample_ids
from header.scoring import final_score
from coarse_search.heading_queries import query_list
from train.checks.heading_results import variant_ids, query_rows, variant_rows, summarise

pipeline = HeadingQueryPipeline(DATA, COARSE, semantic_layer=USE_SEMANTIC_LAYER, semantic_decision=True, gamma=GAMMA,
                                k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY, matcher=MATCHER, min_inliers=MIN_INLIERS, ransac_pixels=RANSAC_PIXELS,
                                matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP, map_margin_m=MAP_MARGIN_M,
                                loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=DINO_WEIGHT,
                                headings=HEADINGS, scale_factors=SCALE_FACTORS)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)

test_set = pipeline.test_set
camera = pipeline.camera

level 3: 0.85 m/pixel, picture 14481 x 9818, map coverage 96%
level 4: 1.20 m/pixel, picture 10240 x 6942, map coverage 96%


map store ready, took 24 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup done in 32.5 s
data: nudt | 500 photos available | zoom levels [3, 4]
photos: nudt | map: NUDT reference map
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
device: cuda
queries per photo: 6 (headings [-5.0, 0.0, 5.0], scale factors [1.0, 1.05]); K and M are per query


In [3]:
import torch
from train.checks.gate_features import gate_row
# which photos: official = all photos in the map window; nudt = a seeded random sample, optionally without the photos of another seed (a fresh, unseen sample)
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEED = os.environ.get("EXCLUDE_SEED")          # e.g. "101": skip the photos that seed 101 picked
n_all = len(test_set.items)
if DATA == "official":
    indices = list(range(n_all)); tag = "official"
else:
    pool = np.arange(n_all)
    if EXCLUDE_SEED:
        seen = set(np.random.default_rng(int(EXCLUDE_SEED)).choice(n_all, N_NUDT, replace=False).tolist())
        pool = np.array([i for i in pool if i not in seen])
    indices = sorted(int(i) for i in np.random.default_rng(NUDT_SEED).choice(pool, min(N_NUDT, len(pool)), replace=False))
    tag = "nudt_seed%d" % NUDT_SEED
DUMP = ROOT / ("tests/Test_Iteration_3/results/fresh_run/dump_%s_conf%.2f" % (tag, MATCHER_CONFIDENCE))
DUMP.mkdir(parents=True, exist_ok=True)
OUT = DUMP / "gate_rows.csv"
done = set(pd.read_csv(OUT).photo) if OUT.exists() else set()
print(tag, len(indices), "photos,", len(done), "already done")

nudt_seed202 200 photos, 0 already done


In [4]:
start = time.time(); failures = []
for count, index in enumerate(indices):
    if index in done:
        continue
    item = test_set.items[index]
    try:
        found = None
        for attempt in range(3):
            try:
                found = pipeline.run_queries(item, test_set.priors[index]); break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); time.sleep(5)
        if found is None:
            raise MemoryError("GPU out of memory three times")
        answer = pipeline.answer_for(found["queries"], [q["id"] for q in found["queries"]])
        row = gate_row(found, answer, test_set.truths[index], test_set.sigmas[index], item["rel_height"])
        row["photo"] = index
        # every candidate of every query, so the decision can be re-made offline in other ways
        arrays = dict(truth=np.asarray(test_set.truths[index], float), sigma=float(test_set.sigmas[index]), height=float(item["rel_height"]), level=int(found["level"]))
        for q, query in enumerate(found["queries"]):
            arrays["q%d_peaks" % q] = np.asarray(query["peaks"], float)
            arrays["q%d_z" % q] = np.asarray(query["z"], float)
            arrays["q%d_inliers" % q] = np.asarray(query["stage2"]["inliers"], float)
            arrays["q%d_shifts" % q] = np.asarray(query["stage2"]["shifts"], float)
            arrays["q%d_heading" % q] = float(query["heading"]); arrays["q%d_scale" % q] = float(query["scale_factor"])
        np.savez_compressed(DUMP / ("photo_%d.npz" % index), **arrays)
        pd.DataFrame([row]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    except Exception as error:
        failures.append((index, repr(error)[:150])); print("photo", index, "failed", repr(error)[:150])
    if count % 10 == 0:
        print("photo %d/%d | %.0f s" % (count, len(indices), time.time() - start), flush=True)
print("done", len(indices), "photos;", len(failures), "failed")

photo 0/200 | 4 s


photo 10/200 | 48 s


photo 20/200 | 91 s


photo 30/200 | 132 s


photo 40/200 | 175 s


photo 50/200 | 216 s


... (11 outputs left out)

photo 170/200 | 719 s


photo 180/200 | 767 s


photo 190/200 | 819 s


done 200 photos; 0 failed
